# 03 — Capa Gold: tablas de negocio
- `gold_registros`: un registro por estación y sismo de Lima, Tainan y Manta, con PGA, PGV, espectro y Vs30.
- `gold_edificios`: un edificio de los 6 distritos por fila, con pisos, periodo, suelo y las proporciones de material del censo.
- `gold_edificios_dano`: edificios de Tainan y Manta con el daño observado.
- `gold_turquia_edificios` y `gold_turquia_shakemap`: edificios de Turquía 2023 con daño observado y el ShakeMap de cada ciudad (solo validación).
- `gold_referencias`: parámetros de la Norma E.030, distritos, microzonificación, suelo no lineal (BSSA14) y fragilidad (GEM).
- `gold_onda_p_ventanas` y `gold_onda_p_picks`: ventanas de STEAD etiquetadas para el detector de onda P.

Se recalculan completas en cada ejecución.

## 1. Setup — imports

In [ ]:
# Librerías que usa el notebook.
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

## 2. Catálogo, esquema y parámetros

In [ ]:
# Catálogo, esquema y zona horaria de la sesión.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

In [ ]:
# Periodos del espectro de respuesta.
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)

## 3. Funciones de escritura

In [ ]:
# Pasa una tabla pandas pequeña a Spark, con los nulos bien tratados.
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


# Guarda un DataFrame como tabla Delta y cuenta sus filas.
def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


# Filas y tamaño de las tablas con un prefijo.
def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append((t.tableName, spark.table(t.tableName).count(), round((d["sizeInBytes"] or 0) / 1e6, 1)))
    return spark.createDataFrame(filas, "tabla string, filas long, mb double")

## 4. Registros de Lima, Tainan y Manta

In [ ]:
# Registros de Lima, Tainan y Manta con las mismas columnas.
COLS_REGISTRO = ["ciudad", "evento", "archivo_estacion", "estacion", "lat", "lon", "vs30", "ev_lat", "ev_lon", "ev_prof_km", "dist_hipo_km",
                 "pga_geo_g", "pgv_geo_cms", "tp_s", "sa_geo", "arias_ms", "d595_s"]
periodos_col = F.array(*[F.lit(float(p)) for p in PERIODOS]).alias("periodos_s")
lima = spark.table("silver_ondas_registros").withColumn("ciudad", F.lit("Lima")).withColumn("arias_ms", F.lit(None).cast("double")).withColumn("d595_s", F.lit(None).cast("double"))
otras = (spark.table("silver_dano_registros").withColumn("ev_lat", F.lit(None).cast("double")).withColumn("ev_lon", F.lit(None).cast("double"))
         .withColumn("ev_prof_km", F.lit(None).cast("double")).withColumn("dist_hipo_km", F.lit(None).cast("double")))
escribir(lima.select(*COLS_REGISTRO, periodos_col).unionByName(otras.select(*COLS_REGISTRO, periodos_col)), "gold_registros", particion="ciudad")

## 5. Edificios de los 6 distritos

In [ ]:
# Edificios de Lima con la tipología del censo de su distrito.
censo = spark.table("silver_tipologia_censo").select("ubigeo", "p_ladrillo", "p_adobe", "p_quincha", "p_madera")
edificios_gold = (spark.table("silver_edificios").drop("geometry_wkt", "tesela", "altura_max_m", "pisos_25d", "niveles_osm", "altura_ultimo_anio_m", "n_anios_altura")
                  .join(F.broadcast(censo), "ubigeo", "left").fillna(0.0, subset=["p_ladrillo", "p_adobe", "p_quincha", "p_madera"]))
escribir(edificios_gold, "gold_edificios", particion="distrito")

## 6. Edificios con daño observado

In [ ]:
# Edificios con daño observado.
escribir(spark.table("silver_dano_edificios"), "gold_edificios_dano")

## 7. Turquía 2023 (validación externa)
Edificios de Antakya, Nurdağı y Jindires con su daño observado y el ShakeMap recortado alrededor de cada ciudad (±0.05°). Solo existen si `01_bronze` cargó los datos de Turquía.

In [ ]:
# Edificios de Turquía y ShakeMap alrededor de cada ciudad.
if spark.catalog.tableExists("silver_turquia_edificios") and spark.catalog.tableExists("silver_turquia_shakemap"):
    MARGEN_SM = 0.05
    edif_tr = spark.table("silver_turquia_edificios")
    escribir(edif_tr, "gold_turquia_edificios")
    cajas_tr = edif_tr.groupBy("ciudad").agg((F.min("lon") - MARGEN_SM).alias("x0"), (F.min("lat") - MARGEN_SM).alias("y0"),
                                            (F.max("lon") + MARGEN_SM).alias("x1"), (F.max("lat") + MARGEN_SM).alias("y1"))
    dentro = F.col("lon").between(F.col("x0"), F.col("x1")) & F.col("lat").between(F.col("y0"), F.col("y1"))
    escribir(spark.table("silver_turquia_shakemap").join(F.broadcast(cajas_tr), dentro).drop("x0", "y0", "x1", "y1"), "gold_turquia_shakemap", particion="ciudad")
else:
    print("Sin datos de Turquía en silver: no se arma la validación externa")

## 8. Referencias

In [ ]:
# Convierte una tabla en filas tipo-clave-valor en JSON.
def como_referencia(tabla, tipo, clave):
    df = spark.table(tabla)
    return df.select(F.lit(tipo).alias("tipo"), F.concat_ws("|", *[F.col(c).cast("string") for c in clave]).alias("clave"), F.to_json(F.struct(*df.columns)).alias("valor"))


referencias = (como_referencia("silver_e030_parametros", "e030", ["perfil"])
               .unionByName(como_referencia("silver_distritos", "distritos", ["ubigeo"]))
               .unionByName(como_referencia("silver_microzonificacion_periodos", "microzonificacion_periodos", ["distrito", "pagina"]))
               .unionByName(como_referencia("silver_bssa14_no_lineal", "bssa14", ["T_s"]))
               .unionByName(como_referencia("silver_fragilidad_parametros", "fragilidad", ["tipologia", "pisos_gem"])))
escribir(referencias, "gold_referencias", particion="tipo")
spark.table("gold_referencias").groupBy("tipo").count().show(truncate=False)

## 9. Detector de onda P

In [ ]:
# Ventanas y picks de STEAD para el detector de onda P.
escribir(spark.table("silver_stead_ventanas"), "gold_onda_p_ventanas")
escribir(spark.table("silver_stead_picks"), "gold_onda_p_picks")

## 10. Verificación

In [ ]:
# Inventario de las tablas gold.
display(inventario("gold_"))